In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px

Load the dataset
---

In [ ]:
dataset = Path.home() / "OneDrive - Microsoft" / "Benchmark" / "Datasets" / "Re-DocRED" / "Extractions" / "2024-07-31" / "full" / "re_docred_extraction_dataset.jsonl"

examples = []
entities = []
with open(dataset, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        entities.extend(row["entities"])
        examples.append(row)

In [ ]:
print(f"Number of examples: {len(examples)}")
print(f"Number of entities: {len(entities)}")
print("Sample entity:")
print(entities[0])
print("Sample example:")
print(examples[0])

Dataset statistics
---

Convert to DataFrame

In [ ]:
df = pd.DataFrame(examples)
df = df.join(pd.json_normalize(df["document"])).drop(columns=["document"])
print(df.columns)

edf = pd.DataFrame(entities)
edf = edf.join(pd.json_normalize(edf["properties"])).drop(columns=["properties"])
edf = edf.drop(columns=["source_ids", "evidence_map", "entity_id"])
print(edf.columns)

In [ ]:
# Unique entity types stats
entity_types = edf["type"].apply(tuple).value_counts().sort_values(ascending=False)
print("Unique types: ", len(entity_types))
entity_types

In [ ]:
# Unique properties
properties = edf.count().sort_values(ascending=False)
properties

In [ ]:
# Add counts
edf["names_count"] = edf["name"].apply(lambda x: len(x))
edf["properties_count"] = edf.apply(lambda row: row.count(), axis=1)

df["entities_count"] = df["entities"].apply(len)
df["text_len"] = df["data.text"].apply(len)

print(df.shape)
print(edf.shape)

In [ ]:
print(f"Number of examples: {len(df):,d}")
print(f"Minimum number of entities per example: {df["entities_count"].min():.2f}")
print(f"Maximum number of entities per example: {df["entities_count"].max():.2f}")
print(f"Average number of entities per example: {df["entities_count"].mean():.2f}")
print(f"Minimum text length of examples (in characters): {df["text_len"].min():.2f}")
print(f"Maximum text length of examples (in characters): {df["text_len"].max():.2f}")
print(f"Average text length of examples (in characters): {df["text_len"].mean():.2f}")
print()
print(f"Number of entities: {len(edf):,d}")
print(f"Minimum number of names per entity: {edf["names_count"].min():.2f}")
print(f"Maximum number of names per entity: {edf["names_count"].max():.2f}")
print(f"Average number of names per entity: {edf["names_count"].mean():.2f}")
print(f"Minimum number of properties per entity: {edf["properties_count"].min():.2f}")
print(f"Maximum number of properties per entity: {edf["properties_count"].max():.2f}")
print(f"Average number of properties per entity: {edf["properties_count"].mean():.2f}")

Entity statistics

In [ ]:
# Histogram of entities by number of properties
take = 10
fig = px.histogram(x=edf["properties_count"].clip(upper=take-1), nbins=2 * take, title="Entities by number of properties",
                   labels={"x": "number of properties", "y": "number of entities"})

total_entities = len(edf)
groups = sorted(edf["properties_count"].unique())[:take]
for i in groups:
    count = (edf["properties_count"] == i).sum() if i != groups[-1] else (edf["properties_count"] >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total_entities:.1%}",
                       showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="entity count")
fig.update_layout(width=1000)
fig.show()

In [ ]:
# Histogram of entities by number of names
take = 5
fig = px.histogram(x=edf["names_count"].clip(upper=take-1), nbins=2 * take, title="Entities by number of names",
                   labels={"x": "number of names", "y": "number of entities"})

total_entities = len(edf)
groups = sorted(edf["names_count"].unique())[:take]
for i in groups:
    count = (edf["names_count"] == i).sum() if i != groups[-1] else (edf["names_count"] >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total_entities:.1%}",
                       showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="entity count")
fig.update_layout(width=1000)
fig.show()

Example statistics

In [ ]:
# Histogram of examples by number of entities
take = 40
nbins = 2 * take
fig = px.histogram(x=df["entities_count"].clip(upper=take-1), nbins=nbins, title="Examples by number of entities",
                   labels={"x": "number of entities", "y": "number of examples"})

total_examples = len(df)
groups = sorted(df["entities_count"].unique())[:take]
for i in groups:
    count = (df["entities_count"] == i).sum() if i != groups[-1] else (df["entities_count"] >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total_examples:.1%}",
                       showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="example count")
fig.update_layout(width=1000)
fig.show()